In [93]:
import utils
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import sys, os
import geopandas as gpd
import cartopy.crs as ccrs
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
# from models.Carleton2022.model.mortality_functions import ReadOUTFiles

IMAGE = {
    "World": "World",
    "CAN":"Canada", "USA":"USA", "MEX":"Mexico", "RCAM":"Central America", "BRA":"Brazil", "RSAM":"Rest of South America", # America
    "NAF":"Northern Africa", "WAF":"Western Africa", "EAF":"Eastern Africa", "SAF":"South Africa", # Africa
    "WEU":'Western Europe', "CEU":"Central Europe", "TUR":"Turkey", "UKR":"Ukraine", # Europe
    "STAN":"Central Asia", "RUS":"Russia region", "ME":"Middle East", "INDIA":"India", "KOR":"Korea region", "CHN":"China region", "SEAS":"Southeastern Asia", "INDO":"Indonesia region", "JAP":"Japan", # Asia
    "OCE":"Oceania", "RSAS":"Rest of South Asia", "RSAF":"Rest of Southern Africa" # Oceania + other
}

### Regression with climate indices

#### Using Present Day ERA5

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"

IOD = xr.open_dataset(era5dir+"IOD_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
NAO = xr.open_dataset(era5dir+"winter_NAO_ERA5_historical_r1i1p1f1_gn_1940-202512.nc")

wdir = "X:/user/liprandicn/mt-comparison/"
regions = ["World", "RSAM", "WEU", "SEAS", "INDIA", "INDO", "OCE"]

def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/DEFAULT/mortality_default_SSP2_ERA5_2000-2025", range(2000,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial

niño = N34.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values - 273.15
dipole = IOD.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values
oscillation = NAO.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).msl.values

indices = {"Niño": niño, "NAO": oscillation, "IOD": dipole}

fig, ax = plt.subplots(7,3, figsize=(12,20))

for i,region in enumerate(regions):    
    for j, index in enumerate(indices.keys()):
    
        ax[i,j].scatter(indices[index], DetrendedMortality(region), color="navy")    
        ax[i,j].plot(indices[index], np.poly1d(np.polyfit(indices[index], DetrendedMortality(region), 1))(indices[index]), color="navy", alpha=0.5)    
        ax[i,j].text(0.05, 0.95, f"r={np.corrcoef(indices[index], DetrendedMortality(region))[0,1]:.2f}", transform=ax[i,j].transAxes, fontsize=10, 
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax[0,j].set_title(index, fontsize=12)
        
    ax[i,0].set_ylabel(region)
    
plt.show()

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
wdir = "X:/user/liprandicn/mt-comparison/"
niño = N34.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values - 273.15


def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/DEFAULT/mortality_default_SSP2_ERA5_2000-2025", range(2000,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial


fig, ax = plt.subplots(5,6, figsize=(20,15))
axs = ax.flatten()

for i,region in enumerate(IMAGE):    
    
    axs[i].scatter(niño, DetrendedMortality(region))    
    axs[i].plot(niño, np.poly1d(np.polyfit(niño, DetrendedMortality(region), 1))(niño), alpha=0.5)    
    axs[i].text(0.05, 0.95, f"r={np.corrcoef(niño, DetrendedMortality(region))[0,1]:.2f}", transform=axs[i].transAxes, fontsize=10, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axs[i].set_title(region, fontsize=12)
        
    # ax[i,0].set_ylabel(region)
    
plt.show()

#### Using historical data 1970-2025

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"

IOD = xr.open_dataset(era5dir+"IOD_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
NAO = xr.open_dataset(era5dir+"winter_NAO_ERA5_historical_r1i1p1f1_gn_1940-202512.nc")

wdir = "X:/user/liprandicn/mt-comparison/"
regions = ["World", "RSAM", "WEU", "SEAS", "INDIA", "INDO", "OCE"]

def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/modes/IMAGE/MOR_modes_SSP2_ERA5_NoAdap_1970-2025", range(1970,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial

niño = N34.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).sst.values - 273.15
dipole = IOD.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).sst.values
oscillation = NAO.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).msl.values

indices = {"Niño": niño, "NAO": oscillation, "IOD": dipole}

fig, ax = plt.subplots(7,3, figsize=(12,20))

for i,region in enumerate(regions):    
    for j, index in enumerate(indices.keys()):
    
        ax[i,j].scatter(indices[index], DetrendedMortality(region), color="navy")    
        ax[i,j].plot(indices[index], np.poly1d(np.polyfit(indices[index], DetrendedMortality(region), 1))(indices[index]), color="navy", alpha=0.5)    
        ax[i,j].text(0.05, 0.95, f"r={np.corrcoef(indices[index], DetrendedMortality(region))[0,1]:.2f}", transform=ax[i,j].transAxes, fontsize=10, 
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax[0,j].set_title(index, fontsize=12)
        
    ax[i,0].set_ylabel(region)
    
plt.show()

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
wdir = "X:/user/liprandicn/mt-comparison/"
niño = N34.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).sst.values - 273.15


def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/modes/IMAGE/MOR_modes_SSP2_ERA5_NoAdap_1970-2025", range(1970,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial


fig, ax = plt.subplots(5,6, figsize=(25,15))
axs = ax.flatten()

for i,region in enumerate(IMAGE.keys()):    
    
    axs[i].scatter(niño, DetrendedMortality(region))    
    axs[i].plot(niño, np.poly1d(np.polyfit(niño, DetrendedMortality(region), 1))(niño), alpha=0.5)    
    axs[i].text(0.05, 0.95, f"r={np.corrcoef(niño, DetrendedMortality(region))[0,1]:.2f}", transform=axs[i].transAxes, fontsize=10, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axs[i].set_title(IMAGE[region], fontsize=12)
    if i%6 == 0:
        axs[i].set_ylabel("Deaths per 100,000 people")
    if i>=21:
        axs[i].set_xlabel("N3.4 SST anomaly (°C)")

plt.tight_layout()
plt.show()

In [ ]:
mortality = utils.LoadMortality(wdir, "carleton2022/output/modes/IMAGE/MOR_modes_SSP2_ERA5_NoAdap_1970-2025", range(1970,2026), "RCAM", "heat", "relative", "all", None)
mortality.iloc[:,30:45]

### IMAGE MAP

In [46]:
ir2IMAGE = {
    'CAN': ['CAN'],
    'USA': ['SPM', 'USA', 'UMI'],
    'MEX': ['MEX'],
    'RCAM': ['AIA', 'ABW', 'BHS', 'BRB', 'BLZ', 'BMU', 'CYM', 'CRI', 'DMA', 'DOM', 'SLV', 'GRD', 'GLP', 'GTM', 'HTI', 'HND', 'JAM', 'MTQ',
                        'MSR', 'ANT', 'NIC', 'PAN', 'PRI', 'KNA', 'LCA', 'VCT', 'TTO', 'TCA', 'VGB', 'VIR', 'CUB', 'BLM', 'ATG', 'MAF', 'CUW', 'SMX', 
                        'BES', 'CL-'],
    'BRA': ['BRA'],
    'RSAM': ['ARG', 'BOL', 'CHL', 'COL', 'ECU', 'FLK', 'GUF', 'GUY', 'PRY', 'PER', 'SUR', 'URY', 'VEN', 'SGS'],
    'NAF': ['DZA', 'EGY', 'LBY', 'MAR', 'TUN', 'ESH'],
    'WAF': ['BEN', 'BFA', 'CMR', 'CPV', 'CAF', 'TCD', 'COD', 'COG', 'CIV', 'GNQ', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'LBR', 'MLI', 'MRT',
                       'NER', 'NGA', 'STP', 'SEN', 'SLE', 'SHN', 'TGO'],
    'EAF': ['BDI', 'COM', 'DJI', 'ERI', 'ETH', 'KEN', 'MDG', 'MUS', 'REU', 'RWA', 'SYC', 'SOM', 'SDN', 'UGA', 'SSD', 'MYT'],
    'SAF': ['ZAF'],
    'WEU': ['AND', 'AUT', 'BEL', 'DNK', 'FRO', 'FIN', 'FRA', 'DEU', 'GIB', 'GRC', 'ISL', 'IRL', 'ITA', 'LIE', 'LUX', 'MLT', 'MCO', 'NLD', 
                       'NOR', 'PRT', 'SMR', 'ESP', 'SWE', 'CHE', 'GBR', 'VAT', 'SJM', 'IMN', 'JEY', 'ALA', 'GGY'],
    'CEU': ['ALB','BIH','BGR','HRV', 'CYP', 'CZE', 'EST', 'HUN', 'LVA', 'LTU', 'MKD', 'POL', 'ROU', 'SRB', 'SVK', 'SVN', 'MNE','KO-'],
    'TUR': ['TUR'],
    'UKR': ['BLR', 'MDA', 'UKR'],
    'STAN': ['KAZ', 'KGZ', 'TJK', 'TKM', 'UZB'],
    'RUS': ['ARM', 'AZE', 'GEO', 'RUS'],
    'ME': ['BHR', 'IRN', 'IRQ', 'ISR', 'JOR', 'KWT', 'LBN', 'OMN', 'QAT', 'SAU', 'SYR', 'ARE', 'YEM', 'PSE'],
    'INDIA': ['IND'],
    'KOR': ['PRK', 'KOR'],
    'CHN': ['CHN', 'HKG', 'MAC', 'MNG', 'TWN'],
    'SEAS': ['BRN', 'KHM', 'LAO', 'MYS', 'MMR', 'PHL', 'SGP', 'THA', 'VNM', 'SP-'],
    'INDO': ['TLS', 'IDN', 'PNG', 'GUM', 'CXR'],
    'JAP': ['JPN'],
    'OCE': ['ASM', 'AUS', 'COK', 'FJI', 'PYF', 'KIR', 'MHL', 'FSM', 'NRU', 'NCL', 'NZL', 'NIU', 'MNP', 'PLW', 'PCN', 'WSM', 'SLB', 'TKL', 'TON',
                'TUV', 'VUT', 'VUT', 'WLF', 'HMD', 'CCK', 'NFK', 'ATF'],
    'RSAS': ['AFG', 'BGD', 'BTN', 'MDV', 'NPL', 'PAK', 'LKA', 'IOT'],
    'RSAF': ['AGO', 'BWA', 'LSO', 'MWI', 'MOZ', 'NAM', 'SWZ', 'TZA', 'ZMB', 'ZWE', 'BVT'], 
    'Greenland': ['GRL'],
    'Antarctica': ['ATA'],
    'Caspian Sea': ['CA-']
}


IMAGE = [
    "World",
    "CAN", "USA", "MEX", "RCAM", "BRA", "RSAM", # America
    "NAF", "WAF", "EAF", "SAF", # Africa
    "WEU", "CEU", "TUR", "UKR", # Europe
    "STAN", "RUS", "ME", "INDIA", "KOR", "CHN", "SEAS", "INDO", "JAP", # Asia
    "OCE", "RSAS", "RSAF" # Oceania + other
]

In [ ]:
def get_region(hierid):
    iso_code = hierid.split('.')[0]
    iso_to_region = {iso: region for region, isos in ir2IMAGE.items() for iso in isos}
    return iso_to_region.get(iso_code, 'Unknown')

ir_dir = "X:\\user\\liprandicn\\mt-comparison\\carleton2022\\data\\carleton_sm\\ir_shp"
ir= gpd.read_file(ir_dir+"/impact-region.shp")

ir['geometry'] = ir['geometry'].buffer(0)
ir['IMAGE'] = ir['hierid'].apply(get_region)
ir_image = ir.dissolve(by='IMAGE')
ir_image

In [ ]:
ir_image["R2"] =""
for region in IMAGE[1:]:
    ir_image.loc[region, "R2"] = np.corrcoef(niño, DetrendedMortality(region))[0,1]
ir_image.drop(index=["Antarctica", "Caspian Sea", "Greenland", ], inplace=True)
ir_image

In [ ]:
ir_image = ir_image.to_crs("ESRI:54030") 
fig, ax = plt.subplots(figsize=(10, 10))
ir_image.plot(column='R2', cmap="coolwarm", ax=ax, vmin=-0.5)

ax.axis('off')
cbar_ax = fig.add_axes([0.25, 0.3, 0.6, 0.01])
# Crear el colorbar con límites comunes
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=ir_image['R2'].min(), vmax=ir_image['R2'].max()))
sm._A = []
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r'Historical correlation $R^2$ between El Niño and annual mortality from heat', size=10)

plt.savefig("X:\\user\\liprandicn\\mt-comparison\\data/niño_mortality_correlation_map.png", dpi=300, bbox_inches='tight')


In [ ]:
rc = pd.read_csv("X:\\user\\liprandicn\\mt-comparison\\carleton2022\\data\\regions\\region_classification.csv")
rc.IMAGE26.unique()